# Clase 061 — CRISP-DM como framework metodológico

Recorremos las 6 fases de CRISP-DM como un mini proyecto end-to-end ejecutable: de los
business success criteria al deployment, mostrando que iterar es la norma, no la excepción.

Requiere: `numpy`, `pandas`, `scikit-learn`, `joblib`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import joblib, tempfile, os
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score

np.random.seed(42)

## Fase 1 — Business Understanding

Caso: **churn** de una telco. Traducimos el problema de negocio a criterios cuantificables,
distintos de las métricas técnicas.

In [ ]:
business_success_criteria = {
    'reducir_churn_pp': 5,          # bajar churn 5 puntos en 6 meses
    'recall_minimo_churners': 0.70, # capturar >=70% de los que se van
    'precision_minima': 0.50,       # <50% de falsas alarmas es inaceptable
}
for k, v in business_success_criteria.items():
    print(f'  criterio de exito: {k} = {v}')
print('\nestos son de NEGOCIO; accuracy/F1 son tecnicos y vienen despues')

## Fase 2 — Data Understanding (EDA + calidad)

Recolectamos y exploramos. Dataset sintético de clientes con churn ~20% (desbalanceado, como
en la vida real).

In [ ]:
X, y = make_classification(n_samples=4000, n_features=10, n_informative=6,
                           weights=[0.8, 0.2], random_state=42)
cols = [f'feat_{i}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=cols); df['churn'] = y
print('shape:', df.shape, '| NaN:', int(df.isna().sum().sum()))
print('tasa de churn:', f'{df.churn.mean():.1%}', '-> target desbalanceado')

## Fase 3 — Data Preparation (split + pipeline)

Separamos train/test estratificado y armamos el pipeline de preprocesamiento. Todo `.fit()`
solo en train.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(df[cols], df['churn'], test_size=0.25,
                                      stratify=df['churn'], random_state=42)
pre = StandardScaler()
print('train', Xtr.shape, '| test', Xte.shape)
print('proporcion churn -> train', f'{ytr.mean():.3f}', '| test', f'{yte.mean():.3f}')

## Fase 4 — Modeling (candidatos + CV)

Dos candidatos justificados: `LogisticRegression` (interpretable, baseline) y `RandomForest`
(no lineal, fuerte en tabular). Comparamos por CV con `recall` (nos importan los churners).

In [ ]:
candidates = {
    'LogReg': Pipeline([('sc', StandardScaler()),
                        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                           random_state=42, n_jobs=-1),
}
cv_scores = {name: cross_val_score(m, Xtr, ytr, cv=5, scoring='recall').mean()
             for name, m in candidates.items()}
best_name = max(cv_scores, key=cv_scores.get)
print('recall CV por candidato:', {k: round(v, 3) for k, v in cv_scores.items()})
print('modelo elegido:', best_name)

## Fase 5 — Evaluation (contra los business success criteria)

No basta con métricas técnicas: mapeamos a los criterios de la fase 1 y tomamos una decisión
go/no-go.

In [ ]:
best = candidates[best_name].fit(Xtr, ytr)
pred = best.predict(Xte)
rec = recall_score(yte, pred)
prec = precision_score(yte, pred)
print(f'recall churners : {rec:.3f}  (criterio >= {business_success_criteria["recall_minimo_churners"]})')
print(f'precision       : {prec:.3f}  (criterio >= {business_success_criteria["precision_minima"]})')

go = rec >= business_success_criteria['recall_minimo_churners'] and \
     prec >= business_success_criteria['precision_minima']
print('\nDECISION:', 'GO -> desplegar' if go else
      'NO-GO -> iterar (volver a Data Preparation / Modeling)')

## Fase 6 — Deployment + iteración

Serializamos el artefacto y dejamos definido el monitoreo. La flecha externa de CRISP-DM
cierra el ciclo: Deployment → nueva ronda de Business Understanding.

In [ ]:
path = os.path.join(tempfile.gettempdir(), 'churn_model.joblib')
joblib.dump(best, path)
reloaded = joblib.load(path)
assert np.array_equal(reloaded.predict(Xte[:5]), best.predict(Xte[:5]))
print('artefacto serializado en', path)
print('plan de monitoreo: PSI por feature + recall en ventana movil; retraining por drift')

fases = ['Business', 'Data Under.', 'Data Prep', 'Modeling', 'Evaluation', 'Deployment']
tiempo = [10, 10, 60, 10, 5, 5]  # % tipico del esfuerzo
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(fases[::-1], tiempo[::-1], color='#37a')
ax.set_xlabel('% tipico del esfuerzo del proyecto')
ax.set_title('CRISP-DM: Data Preparation domina (50-70%)')
plt.tight_layout(); plt.show()

## Ejercicios

1. Cambiá los `business_success_criteria` (por ejemplo recall ≥ 0.85) y observá cómo la
   decisión de la fase 5 pasa de GO a NO-GO. ¿Qué iteración dispararía eso?
2. Agregá un tercer candidato (`GradientBoostingClassifier`) en la fase 4 y decidí con el
   mismo criterio de recall. ¿Cambia el elegido?
3. Redactá un párrafo por fase para un caso real propio (churn de Netflix, demanda de una
   panadería) con al menos una iteración explícita entre fases.
4. Armá una tabla comparando CRISP-DM, TDSP y el ML lifecycle moderno (MLOps) en 5 filas:
   origen, fases, foco, herramientas, cuándo usarlo.

## Conclusiones

- CRISP-DM ordena el trabajo en 6 fases; Data Preparation consume el 50-70% del esfuerzo.
- Los business success criteria se definen ANTES de modelar y mandan en la fase de Evaluation.
- No es cascada: las flechas son bidireccionales y hay un loop externo (iterar es la norma).
- La fase de Evaluation mapea a criterios de negocio, no solo a accuracy/F1 técnicos.